### Datos Stream desde archivos en la nube (Auto Loader)

1. Leer archivos de la "nube" utilizando API - DataStreamReader
2. Transformar el DataFrame y agregar las siguientes columnas:
    - file path: Ruta del archivo en la nube
    - ingestion date: Current Timestamp
3. Escribir los datos de Stream transformados en Tabla Delta Lake

#### 1. Leer archivos de la "nube" utilizando API - DataStreamReader

In [0]:
department_df = (
     spark.readStream
     .format("cloudFiles")
     .option("cloudFiles.format", "json")
     .option("clodFiles.useNotifications", "false")
     .option("cloudFiles.schemaLocation", "/Volumes/zenviro/raw/operational_data/departments_autoloader/_schema")
     .option("cloudFiles.inferColumnTypes", "true")
     .option("cloudFiles.schemaHints", "department_id integer, location_id integer")
     .option("cloudFiles.schemaEvolutionMode", "rescue")
     .load("/Volumes/zenviro/raw/operational_data/departments_autoloader/")
)

#### 2. Transformar el DataFrame y agregar las siguientes columnas:
- file path: Ruta del archivo en la nube
- ingestion date: Current Timestamp

In [0]:
from pyspark.sql.functions import current_timestamp, col

department_transformed_df = (
  department_df.withColumn('file_path', col("_metadata.file_path"))
             .withColumn("ingestion_date", current_timestamp())
)

#### 3. Escribir los datos de Stream transformados en Tabla Delta Lake

In [0]:
streaming_query = ( 
  department_transformed_df.writeStream
  .format("delta")
  .option("checkpointLocation", "/Volumes/zenviro/raw/operational_data/departments_autoloader/_checkpoint_stream")
  .trigger(availableNow=True)
  .toTable("zenviro.bronze.departments_autoloader")
)

In [0]:
%sql
SELECT * FROM zenviro.bronze.departments_autoloader;

In [0]:
%sql
SELECT file_path, count(1)
FROM zenviro.bronze.departments_autoloader
GROUP BY file_path;